# Local and open models
**Ollama in depth · where open weights come from · quantization · tool calling on small models · local vs hosted speed**




## 1. Understanding Local Models and Ollama

A **local model** is an AI model that runs on my own computer instead of sending every request to a cloud-based AI service.

**Ollama** is a tool that helps me download and run AI models locally.

benefits of local models:
- Privacy: Data can remain on my computer when the entire workflow is configured to run locally.
- Offline use: After downloading the model, I can generally use it without an internet connection.
- Cost: I may avoid per-request API charges, although my computer still uses electricity and hardware resources.
- Control: I can choose which supported models to download and run.

**limitations**
- Local models need enough RAM or GPU memory.
- They may respond more slowly than hosted models.
- Smaller models may give less accurate answers or struggle with complex tasks.
- Main idea: Local models give me more control, while hosted models can 

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from local_models_utils import model_size_gb, fits_on, laptop_options, load_local_catalog, probe_tool_calling, speed_table
from providers_utils import groq_client, ollama_client, ask
from landscape_utils import time_stream, PROMPT

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")
try:
    ollama = ollama_client()
    local_models = [m.id for m in ollama.models.list().data]
    print("Ollama models available:", local_models)
except Exception as e:
    ollama, local_models = None, []
    print("Ollama not reachable:", e)

Ollama models available: ['llama3.2:3b', 'qwen2.5:7b']


## 2. Where open models come from: Hugging Face
Ollama does not train models — it packages and runs models published on hubs like Hugging Face, the same place Groq's hosted open-weight models originate from. The names differ slightly: a model on Hugging Face as `meta-llama/Llama-3.2-3B-Instruct` becomes `llama3.2:3b` as an Ollama tag, and `openai/gpt-oss-20b` on Groq is the same underlying weights as `gpt-oss:20b` if you pulled it locally.

*Question 1 — If a Hugging Face model page says "gated: request access", what does that tell you about its licence?*

In [2]:
print(load_local_catalog())

[{'id': 'llama32-1b', 'name': 'llama3.2:1b', 'params_billion': 1.0, 'precision': 'int4', 'tool_calling': False, 'notes': 'Smallest useful chat model. Fast on anything, weak reasoning.'}, {'id': 'llama32-3b', 'name': 'llama3.2:3b', 'params_billion': 3.0, 'precision': 'int4', 'tool_calling': False, 'notes': 'The course default local model. No reliable tool calling.'}, {'id': 'qwen25-7b', 'name': 'qwen2.5:7b', 'params_billion': 7.0, 'precision': 'int4', 'tool_calling': True, 'notes': 'Supports tool calling reasonably well for its size.'}, {'id': 'llama31-8b', 'name': 'llama3.1:8b', 'params_billion': 8.0, 'precision': 'int4', 'tool_calling': True, 'notes': 'Good general-purpose local model if RAM allows.'}, {'id': 'gpt-oss-20b-local', 'name': 'gpt-oss:20b (local)', 'params_billion': 20.0, 'precision': 'int4', 'tool_calling': True, 'notes': 'Same weights as the hosted Groq default. Needs a capable laptop or a small GPU.'}, {'id': 'gpt-oss-120b-local', 'name': 'gpt-oss:120b (local)', 'params

## 3. Quantization: what it actually costs in memory
Quantization is a technique that reduces the precision used to store a model's parameters.

To understand this, I first need to know what a parameter is.

A parameter is a learned numerical value inside an AI model. Models can have millions or billions of parameters.

Each parameter needs memory to store its value. The amount of memory depends partly on how many bits are used to represent it.

In [3]:
print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")
for prec in ("fp32", "fp16", "int8", "int4"):
    print(f"{prec:10} {'':>12} {model_size_gb(20, prec):>12.1f} GB")

precision   bytes/param    a 20B model
fp32                            80.0 GB
fp16                            40.0 GB
int8                            20.0 GB
int4                            10.0 GB


In [4]:
# Which catalog models fit YOUR laptop? Edit this to your real RAM.
MY_RAM_GB = 16
fits = laptop_options(MY_RAM_GB)
print(f"On a {MY_RAM_GB} GB laptop, these fit (with 2 GB headroom for everything else):")
for m in fits:
    print(f"  {m['name']:22} {model_size_gb(m['params_billion'], m['precision']):5.1f} GB   tool calling: {m['tool_calling']}")

On a 16 GB laptop, these fit (with 2 GB headroom for everything else):
  llama3.2:1b              0.5 GB   tool calling: False
  llama3.2:3b              1.5 GB   tool calling: False
  qwen2.5:7b               3.5 GB   tool calling: True
  llama3.1:8b              4.0 GB   tool calling: True
  gpt-oss:20b (local)     10.0 GB   tool calling: True


*Question 2 — Why does the table use int4 sizes for the catalog instead of fp16, when fp16 is what the model was trained at?*

## 4. Checking Whether a Model Fits My Laptop
Before downloading a model, I should check whether my computer has enough memory to run it.



Before downloading a model, I should check whether my computer has enough memory to run it.

- The required memory depends on:
- Number of parameters in the model.
- Precision or quantization format.
- Memory needed for the input and generated response.
- Other applications running on my computer.
- Whether the model uses my GPU, system RAM, or both.

**My model-selection process**

- Check my available RAM and GPU memory.
- Check the model's parameter count and quantization.
- Leave some memory available for other tasks.
- Test the model and measure its performance.

In [5]:
if ollama and local_models:
    for name in local_models[:3]:
        r = probe_tool_calling(ollama, name)
        print(f"{name:20} supports_tool_calling={r['supports_tool_calling']}")
        if not r["supports_tool_calling"]:
            print("   said instead:", (r.get('raw_text') or '')[:150])
else:
    print("skipped (Ollama not running)")

llama3.2:3b          supports_tool_calling=True
qwen2.5:7b           supports_tool_calling=True


*Question 3 — If your local model fails the tool-calling probe, what are your two options for Module 6's agent labs?*

## 5. Local vs hosted: measure, don't guess
A hosted model runs on a provider's infrastructure and is accessed through an API.

A **local model** runs on my own computer.

Both can answer questions, but their performance may be different.

What should I compare?

- Speed: How quickly does the model generate responses?
- Tokens per second: How many output tokens does it generate each second?
- Privacy: Where is my data processed?
- Cost: Are there API charges, hardware costs, or both?
- Quality: Does the model answer correctly?
- Tool calling: Can it reliably use the tools required by my application?

**Which model should I choose?**

It depends on the job.
- FAQ bot: A local model may be suitable if it answers accurately and its speed is acceptable.
- Document reader: A local model may be useful when documents contain sensitive information and the workflow must remain on the computer.
- Agent assistant: A hosted model may be preferable if it provides more reliable tool calling and faster responses.

These are starting points, not fixed rules. I should test each option against the project's requirements.

In [6]:
results = {}
try:
    results["groq"] = time_stream(groq, GROQ_MODEL)
except Exception as e:
    print("groq failed:", e)
if ollama and local_models:
    try:
        results["ollama"] = time_stream(ollama, local_models[0])
    except Exception as e:
        print("ollama failed:", e)
else:
    print("ollama skipped (not running)")

if results:
    print(speed_table(results))

provider     TTFT s   total s   tokens    tok/s
groq           3.28      3.35     1433    427.4
ollama         3.67     22.92      139      6.1


## 6. Reflection
Answer questions 1-4 above, one or two lines each.

1.
2.
3.
4.

Then run the checkpoint: `uv run pytest tests/test_s12.py`